In [1]:
import torch
import torch.nn as nn

In [2]:
class Model(nn.Module):
  def __init__(self, nums_feature):
    super().__init__()
    self.linear = nn.Linear(nums_feature, 1) # z= w1x1+w2x2+b  # (input, output)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):
    z =  self.linear(features)
    out = self.sigmoid(z)
    return out

In [3]:
# create dataset
features = torch.rand(20, 3)

# model
model = Model(features.shape[1])
# model.forwarded(features)
model(features)

tensor([[0.4910],
        [0.5590],
        [0.5188],
        [0.5511],
        [0.5089],
        [0.4909],
        [0.5660],
        [0.4794],
        [0.5760],
        [0.5416],
        [0.5684],
        [0.5230],
        [0.5418],
        [0.5593],
        [0.5808],
        [0.4997],
        [0.4841],
        [0.5081],
        [0.4868],
        [0.5042]], grad_fn=<SigmoidBackward0>)

In [4]:
# show model weight
model.linear.weight

Parameter containing:
tensor([[0.4528, 0.1671, 0.0506]], requires_grad=True)

In [5]:
# show model bias
model.linear.bias

Parameter containing:
tensor([-0.2149], requires_grad=True)

In [6]:
!pip install torchinfo

In [7]:
from torchinfo import summary
summary(model, input_size=(features.shape[0], features.shape[1]))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [20, 1]                   --
├─Linear: 1-1                            [20, 1]                   4
├─Sigmoid: 1-2                           [20, 1]                   --
Total params: 4
Trainable params: 4
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

### model 2- more than one layer

In [22]:
class Model2(nn.Module):
  def __init__(self, nums_feature):
    super().__init__()
    self.linear1 = nn.Linear(3, 2)
    self.relu = nn.ReLU()

    self.linear2 = nn.Linear(2, 1)
    self.sigmoid = nn.Sigmoid()


  def forward(self, features):
    out =  self.linear1(features) # 5
    # print(out)
    out = self.relu(out) # 6
    out = self.linear2(out) # 10
    out = self.sigmoid(out)

    return out

In [23]:
features2 = torch.rand(20, 3)

# model
model2 = Model2(features2.shape[1])
# model.forwarded(features)
model2(features2)

tensor([[0.6370],
        [0.6164],
        [0.6362],
        [0.6222],
        [0.6202],
        [0.6130],
        [0.6532],
        [0.6224],
        [0.5537],
        [0.6125],
        [0.6312],
        [0.6711],
        [0.5803],
        [0.6113],
        [0.6211],
        [0.5857],
        [0.5844],
        [0.6026],
        [0.6599],
        [0.6552]], grad_fn=<SigmoidBackward0>)

In [24]:
summary(model2, input_size=(features2.shape[0], features2.shape[1]))

Layer (type:depth-idx)                   Output Shape              Param #
Model2                                   [20, 1]                   --
├─Linear: 1-1                            [20, 2]                   8
├─ReLU: 1-2                              [20, 2]                   --
├─Linear: 1-3                            [20, 1]                   3
├─Sigmoid: 1-4                           [20, 1]                   --
Total params: 11
Trainable params: 11
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

model 3- using sequential

In [11]:
class Model3(nn.Module):
  def __init__(self, nums_feature):
    super().__init__()
    self.networks = nn.Sequential(
        nn.Linear(3, 2),
        nn.ReLU(),
        nn.Linear(2, 1),
        nn.Sigmoid()
    )

  def forward(self, features):
    out =  self.networks(features)
    return out

In [25]:
features3 = torch.rand(20, 3)

# model
model3 = Model3(features3.shape[1])
# model.forwarded(features)
model3(features3)

tensor([[0.3815],
        [0.3791],
        [0.3657],
        [0.3943],
        [0.3834],
        [0.3544],
        [0.3751],
        [0.3713],
        [0.3682],
        [0.3559],
        [0.3697],
        [0.3583],
        [0.3652],
        [0.3869],
        [0.3703],
        [0.3633],
        [0.3625],
        [0.3676],
        [0.3693],
        [0.3778]], grad_fn=<SigmoidBackward0>)

In [13]:
summary(model3, input_size=(features3.shape[0], features3.shape[1]))

Layer (type:depth-idx)                   Output Shape              Param #
Model3                                   [20, 1]                   --
├─Sequential: 1-1                        [20, 1]                   --
│    └─Linear: 2-1                       [20, 2]                   8
│    └─ReLU: 2-2                         [20, 2]                   --
│    └─Linear: 2-3                       [20, 1]                   3
│    └─Sigmoid: 2-4                      [20, 1]                   --
Total params: 11
Trainable params: 11
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [14]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

columns = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"
]

df = pd.read_csv(url, names=columns)

print(df.head())
df.shape

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  


(768, 9)

In [15]:
# Separate features and labels
X = df.iloc[:,:-1]
y= df.iloc[:,-1]

# Train test split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [17]:
# Convert to tensors
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()

y_train_tensor = torch.from_numpy(y_train.values).float().view(-1,1)
y_test_tensor = torch.from_numpy(y_test.values).float().view(-1,1)

In [18]:
class OurNN(nn.Module):
  def __init__(self, input_size):
    #intialize wight and bias
    super().__init__()
    # self.weights= torch.rand(input_size, 1, requires_grad=True)
    # self.bias= torch.zeros(1, requires_grad=True)
    self.linear = nn.Linear(input_size, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, X):
    #forward pass
    z = self.linear(X)
    out = self.sigmoid(z)
    return out

  def loss_function(self,y_pred, y):
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)
    loss = -(y*torch.log(y_pred) + (1-y)*torch.log(1-y_pred))
    return loss.mean()

In [19]:
#hyperparameters

learning_rate = 0.01
epochs = 5000


# creating model instance
model = OurNN(X_train_tensor.shape[1])

#training loop

for epoch in range(epochs):

    # 1. Forward pass
    y_pred = model(X_train_tensor)

    # 2. Loss calculation
    loss = model.loss_function(y_pred, y_train_tensor)

    # 3. Backward pass
    loss.backward()

    # 4. Update parameters
    with torch.no_grad():
        model.linear.weight -= learning_rate * model.linear.weight.grad
        model.linear.bias -= learning_rate * model.linear.bias.grad

    # 5. Zero gradients
    model.linear.weight.grad.zero_()
    model.linear.bias.grad.zero_()
    print(f"Epoch : {epoch+1} , Loss : {loss.item()}")

Epoch : 1 , Loss : 0.7288579940795898
Epoch : 2 , Loss : 0.7274543642997742
Epoch : 3 , Loss : 0.7260611057281494
Epoch : 4 , Loss : 0.7246782779693604
Epoch : 5 , Loss : 0.7233056426048279
Epoch : 6 , Loss : 0.7219432592391968
Epoch : 7 , Loss : 0.7205909490585327
Epoch : 8 , Loss : 0.7192486524581909
Epoch : 9 , Loss : 0.7179164886474609
Epoch : 10 , Loss : 0.7165939807891846
Epoch : 11 , Loss : 0.7152813673019409
Epoch : 12 , Loss : 0.7139785289764404
Epoch : 13 , Loss : 0.7126852869987488
Epoch : 14 , Loss : 0.7114015817642212
Epoch : 15 , Loss : 0.7101272940635681
Epoch : 16 , Loss : 0.7088624835014343
Epoch : 17 , Loss : 0.707607090473175
Epoch : 18 , Loss : 0.7063607573509216
Epoch : 19 , Loss : 0.7051237225532532
Epoch : 20 , Loss : 0.7038956880569458
Epoch : 21 , Loss : 0.7026767730712891
Epoch : 22 , Loss : 0.7014667391777039
Epoch : 23 , Loss : 0.7002655863761902
Epoch : 24 , Loss : 0.6990732550621033
Epoch : 25 , Loss : 0.6978896260261536
Epoch : 26 , Loss : 0.6967146396636

In [20]:
with torch.no_grad():
    y_pred = model(X_test_tensor)
    y_pred_class = (y_pred>0.5 ).float()
    accuracy = (y_pred_class == y_test_tensor).float().mean()

    print(f"Accuracy : {accuracy.item()}")

Accuracy : 0.7532467246055603
